In [1]:
import pandas as pd
import sqlite3

In [2]:
df = pd.read_csv("../data/loan_customers_cleaned.csv")
df.head()

,customer_id,age,income,loan_amount,tenure,employment_type,credit_score,city,default_flag,credit_score_group,income_group,risk_flag
0,1,25,25000,50000,12,Contract,580,Delhi,1,Poor,Low,High Risk
1,2,35,60000,200000,24,Salaried,720,Mumbai,0,Good,Medium,Low Risk
2,3,40,45000,150000,36,Business,650,Pune,0,Good,Medium,Low Risk
3,4,29,30000,80000,12,Contract,590,Delhi,1,Poor,Low,High Risk
4,5,50,80000,300000,48,Salaried,780,Mumbai,0,Excellent,High,Low Risk


In [3]:
# create sql database
conn = sqlite3.connect("Database/loan_customers_db")

In [4]:
# convert dataframe to sql
df.to_sql("loan_customers_table", conn, if_exists = "replace", index = False)

200

In [6]:
pd.read_sql_query(
    " select * from loan_customers_table where credit_score > 600" , conn
)

,customer_id,age,income,loan_amount,tenure,employment_type,credit_score,city,default_flag,credit_score_group,income_group,risk_flag
0,2,35,60000,200000,24,Salaried,720,Mumbai,0,Good,Medium,Low Risk
1,3,40,45000,150000,36,Business,650,Pune,0,Good,Medium,Low Risk
2,5,50,80000,300000,48,Salaried,780,Mumbai,0,Excellent,High,Low Risk
3,7,32,52000,120000,24,Salaried,710,Chennai,0,Good,Medium,Low Risk
4,8,45,75000,250000,36,Business,740,Hyderabad,0,Good,High,Low Risk
...,...,...,...,...,...,...,...,...,...,...,...,...
163,196,29,35000,96000,18,Contract,630,Delhi,1,Good,Medium,Low Risk
164,197,52,91000,360000,60,Salaried,830,Pune,0,Excellent,High,Low Risk
165,198,37,58000,170000,24,Salaried,725,Mumbai,0,Good,Medium,Low Risk
166,199,44,75000,265000,36,Business,760,Hyderabad,0,Excellent,High,Low Risk


In [13]:
# Defualt Rate
pd.read_sql_query(
    """ select count(*) as total_customers, 
    sum(default_flag) as total_defaults, 
    round(avg(default_flag)*100,2) as default_rate 
    from loan_customers_table """ 
    ,  conn)

,total_customers,total_defaults,default_rate
0,200,50,25.0


In [15]:
# Defualt Rate by employment type
pd.read_sql_query(
    """ select employment_type, count(*) as total_customers, 
    round(avg(default_flag)*100,2) as default_rate 
    from loan_customers_table 
    group by employment_type """ 
    ,  conn)

,employment_type,total_customers,default_rate
0,Business,65,0.00
1,Contract,60,83.33
2,Salaried,75,0.00


In [21]:
# credit score band analysis

query = """
select 
case
    when credit_score < 600 then 'Poor'
    when credit_score between 600 and 700 then 'Average'
    else 'Good'
end as score_band,
count(*) as customers, 
round(avg(default_flag)*100, 2) as default_rate
from loan_customers_table
group by score_band """

pd.read_sql_query(query, conn)


,score_band,customers,default_rate
0,Average,48,50.0
1,Good,126,0.0
2,Poor,26,100.0


In [25]:
# customers per city 
query = """
select city, 
count(*) as total_customers from loan_customers_table
group by city
"""
pd.read_sql_query(query, conn)

,city,total_customers
0,Bangalore,32
1,Chennai,25
2,Delhi,43
3,Hyderabad,30
4,Mumbai,37
5,Pune,33


In [26]:
# customers per city with good credit_score
query = """
select city, 
count(*) as total_customers from loan_customers_table
group by city
having credit_score > 700
"""
pd.read_sql_query(query, conn)

,city,total_customers
0,Chennai,25
1,Hyderabad,30
2,Mumbai,37
